# Finding data sources and signals of interest

The Epidata API includes numerous data streams -- medical claims data, cases and
deaths, wastewater concentrations, and many others -- covering different
geographic regions. This can make it a challenge to find the data stream that
you are most interested in.

Data streams fall into three categories: [V5 sources](#v5-sources),
[migrating endpoints](#migrating-endpoints), and
[historical endpoints](#historical-endpoints) (which include international
sources and private endpoints requiring authentication).

In [ ]:
# Hidden cell (set in the metadata for this cell)
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 1000)

## V5 sources

The V5 API is the primary interface for active epidemiological surveillance
data.

### Online documentation and EpiPortal

The online documentation lists all data sources and signals available through
the [Delphi V5 API](https://cmu-delphi.github.io/delphi-epidata/api/v5_signals.html).

For an interactive visual exploration, the
[Delphi EpiPortal](https://delphi.cmu.edu/epiportal/) lets you filter sources
and signals by disease, pathogen, geography, and date range, view live preview
charts, and copy query code.

### Exploring metadata with `epidata_meta()`

For sources on the V5 API (queried with `epidata_snapshot()` and
`epidata_archive()`), `epidata_meta()` is the primary metadata lookup.

Called with no arguments, it returns a dict keyed by source name covering every
active V5 source:

In [ ]:
from epidatpy import EpiDataContext, EpiRange

epidata = EpiDataContext()

meta = epidata.epidata_meta()
sorted(meta)

Called with a `source`, it returns that source's entry: its available signals,
supported geographic levels, reference date range, and version history (as UTC
timestamps).

In [ ]:
nssp_meta = epidata.epidata_meta(source="nssp")

# all the fields available for this source
sorted(nssp_meta)

In [ ]:
print(nssp_meta["signals"])  # available signal names
print(nssp_meta["geo_types"])  # supported geography levels
print(nssp_meta["reference_time_range"])  # earliest/latest reference_time available
print(nssp_meta["report_time_range"])  # earliest/latest report_time (publication instant) available

You can also flatten the metadata into a data frame to search across all
sources. (A source whose metadata is temporarily unavailable comes back as an
error stub without a `signals` key, so skip those.)

In [ ]:
signals_df = pd.DataFrame(
    [
        {"source": src, "signal": sig, "geo_types": ", ".join(entry["geo_types"])}
        for src, entry in meta.items()
        if "signals" in entry
        for sig in entry["signals"]
    ]
)

# Search for signals related to influenza
signals_df[signals_df["signal"].str.contains("flu", case=False)]

### Example queries for V5 sources

The V5 API uses `epidata_snapshot()` to fetch data as of a given moment (latest
by default) and `epidata_archive()` to fetch the full revision history. See
[Getting started](getting_started.ipynb) for a general introduction to the
package and [Accessing versioned data](versioned_data.ipynb) for details on
versioning.

Both methods take one or more `signals` and one or more `geo_type` values.
`geo_values` and `reference_time` are filtered locally after the request, so
they accept single values, lists, or an `EpiRange`. A `report_time` filter is
applied server-side and takes a comparison string (such as `"<2024-12-15"`) or
an `EpiRange`; a bare date raises, since a single point in time is a
`snapshot_date` question.

Here are examples across several major V5 surveillance streams:

In [ ]:
# NSSP: Influenza emergency department visits, across multiple states
epidata.epidata_snapshot(
    source="nssp",
    signals="pct_ed_visits_influenza",
    geo_type="state",
    geo_values=["pa", "ca"],
    reference_time=EpiRange("2024-10-01", "2024-10-15"),
).df()

In [ ]:
# NHSN: Confirmed hospital admissions, for multiple signals at once
epidata.epidata_snapshot(
    source="nhsn",
    signals=["confirmed_admissions_flu_ew", "confirmed_admissions_covid_ew"],
    geo_type="state",
    geo_values="pa",
    reference_time=EpiRange("2024-10-01", "2024-10-21"),
).df()

In [ ]:
# POPHIVE: Outpatient COVID-19 emergency visits, for a single exact reference date
epidata.epidata_snapshot(
    source="pophive",
    signals="covid_pct_ed",
    geo_type="state",
    geo_values="pa",
    reference_time="2024-10-05",
).df()

In [ ]:
# NWSS: Wastewater SARS-CoV-2 concentrations, for a set of specific dates rather than a range
epidata.epidata_snapshot(
    source="nwss",
    signals="covid_avg_conc",
    geo_type="sewershed",
    geo_values="128",
    reference_time=["2024-12-03", "2024-12-10"],
).df()

In [ ]:
# Archive: Revision history for a single reference date, using a comparison
# operator on report_time instead of an EpiRange
epidata.epidata_archive(
    source="nssp",
    signals="pct_ed_visits_influenza",
    geo_type="state",
    geo_values="pa",
    reference_time="2024-12-07",
    report_time="<2024-12-15",
).df()

## Migrating endpoints

Datasets that originated in the legacy API like `pub_covidcast()`,
`pub_covidcast_meta()`, `pub_fluview()`, `pub_fluview_clinical()`,
`pub_fluview_meta()`, `pub_flusurv()`, and `pub_meta()` are transitioning to
V5. Starting in October 2026, these V4 methods are tentatively deprecated in
favor of V5: their historical data will remain available for at least a year,
but new ingestion will end. Calling them emits a `UserWarning` pointing to the
[migration guide](migration_guide.ipynb). `CovidcastEpidata`, which describes
them, is not being retired outright, but will only keep describing frozen
historical data once a source's V4 ingestion stops.

### Exploring legacy COVIDcast sources with `CovidcastEpidata`

For datasets still queried through the legacy `pub_covidcast()` method,
`CovidcastEpidata` describes all available COVIDcast data sources and signals.
Its `source_df` property mirrors the
[COVIDcast signals documentation](https://cmu-delphi.github.io/delphi-epidata/api/covidcast_signals.html):

In [ ]:
from epidatpy import CovidcastEpidata

covid_sources = CovidcastEpidata()
covid_sources.source_df.head()

This data frame contains the following columns:

- `source` - API-internal source name.
- `name` - Human-readable source name.
- `description` - Description of the source.
- `reference_signal` - The source's headline signal.
- `license` - The license.
- `dua` - Link to the Data Use Agreement.
- `signals` - List of signals available from this data source.

`source_names()` lists the sources directly, and `signal_df` has one row per
signal with its geographic levels, temporal resolution, and descriptive flags
(`is_smoothed`, `is_cumulative`, `has_stderr`, ...):

In [ ]:
covid_sources.source_names()

In [ ]:
covid_sources.signal_df.head()

Index the object with a source name, or a `(source, signal)` pair, to drill in:

In [ ]:
covid_sources["nssp"].signal_df

In [ ]:
covid_sources["fb-survey", "smoothed_cli"]

### Example legacy query

Legacy endpoints remain accessible while their sources transition:

In [ ]:
epidata.pub_covidcast(
    data_source="fb-survey",
    signals="smoothed_accept_covid_vaccine",
    geo_type="state",
    time_type="day",
    time_values=EpiRange(20201221, 20201225),
    geo_values="pa",
).df()

See the [migration guide](migration_guide.ipynb) for the argument mapping from
V4 to V5.

## Historical endpoints

Some datasets are not moving to V5 because data collection has ended. These
endpoints remain available for historical reference using their original
`pub_*` and `pvt_*` methods, and calling one emits a note saying so. For more
information on the datasets that are and are not moving, see
[Endpoints kept for historical reference](migration_guide.ipynb#endpoints-kept-for-historical-reference)
in the migration guide and the
[Delphi V5 Sources and Signals](https://cmu-delphi.github.io/delphi-epidata/api/v5_signals.html)
documentation.

### Exploring package endpoints with `available_endpoints()`

`available_endpoints()` lists all `pub_*` and `pvt_*` endpoint methods in the
package with brief descriptions. Generally, any endpoint listed in the Delphi
Epidata API documentation has an associated method here, named after the API
endpoint with a `pub_` or `pvt_` prefix, e.g. `pub_fluview` or `pvt_twitter`.

In [ ]:
from IPython.display import HTML

from epidatpy import available_endpoints

HTML(available_endpoints().to_html(index=False))

### Examples

Some endpoints contain data within the United States only, some are
international, and some are private and require additional access to query.
Each method's docstring links to its API documentation page.

#### Domestic endpoints

In [ ]:
# Google Flu Trends: Historical flu search volume
epidata.pub_gft(locations="hhs1", epiweeks=EpiRange(201401, 201404)).df()

In [ ]:
# Wikipedia: Article page view counts
epidata.pub_wiki(
    articles="influenza",
    time_type="day",
    time_values=EpiRange(20200101, 20200105),
).df()

In [ ]:
# COVID-19 hospitalizations: State-level timeseries
epidata.pub_covid_hosp_state_timeseries(states="pa", dates=EpiRange(20210101, 20210105)).df()

In [ ]:
# COVID-19 hospitalizations: Facility lookup and by-facility timeseries
epidata.pub_covid_hosp_facility_lookup(city="southlake").df()

In [ ]:
epidata.pub_covid_hosp_facility(hospital_pks="100075", collection_weeks=EpiRange(20200101, 20200501)).df()

In [ ]:
# Delphi's ILINet forecasts (classic JSON format only)
epidata.pub_delphi(system="ec", epiweek=201501)()["epidata"]

In [ ]:
# Delphi's ILI Nearby nowcasts
epidata.pub_nowcast(locations="ca", epiweeks=EpiRange(202201, 202210)).df()

#### International endpoints

In [ ]:
# PAHO Dengue: Surveillance in the Americas
epidata.pub_paho_dengue(regions="ca", epiweeks=EpiRange(202001, 202004)).df()

In [ ]:
# ECDC ILI: Influenza-like illness in Europe
epidata.pub_ecdc_ili(regions="austria", epiweeks=EpiRange(201901, 201904)).df()

In [ ]:
# KCDC ILI: Influenza-like illness in South Korea
epidata.pub_kcdc_ili(regions="ROK", epiweeks=EpiRange(201801, 201804)).df()

In [ ]:
# Taiwan CDC NIDSS: Influenza outpatient visits
epidata.pub_nidss_flu(regions="nationwide", epiweeks=EpiRange(201801, 201804)).df()

In [ ]:
# Taiwan CDC NIDSS: Dengue cases
epidata.pub_nidss_dengue(locations="nationwide", epiweeks=EpiRange(201801, 201804)).df()

In [ ]:
# PAHO Dengue Nowcasts: Delphi nowcast estimates for the Americas
epidata.pub_dengue_nowcast(locations="ca", epiweeks=EpiRange(201501, 201504)).df()

#### Private endpoints

Some private endpoints require a dedicated secret key passed via the `auth`
argument (separate from the standard Epidata API key). Store these in
environment variables or a `.env` file rather than in code. These examples are
not executed here:

```python
import os

# CDC Web Metrics: Website traffic for select topics
epidata.pvt_cdc(
    auth=os.environ["SECRET_API_AUTH_CDC"],
    locations="ma",
    epiweeks=EpiRange(202003, 202304),
).df()

# Digital Surveillance Sensors: Delphi sensor estimates
epidata.pvt_sensors(
    auth=os.environ["SECRET_API_AUTH_SENSORS"],
    names="delphi",
    locations="nat",
    epiweeks=EpiRange(202001, 202010),
).df()

# Twitter / HealthTweets: Influenza and total tweet counts
epidata.pvt_twitter(
    auth=os.environ["SECRET_API_AUTH_TWITTER"],
    locations="hhs1",
    time_type="day",
    time_values=EpiRange(20200101, 20200115),
).df()

# Google Health Trends: Search queries
epidata.pvt_ght(
    auth=os.environ["SECRET_API_AUTH_GHT"],
    locations="ca",
    query="cough",
    epiweeks=EpiRange(202001, 202003),
).df()

# CDC NoroSTAT: Norovirus outbreak data
epidata.pvt_norostat(
    auth=os.environ["SECRET_API_AUTH_NOROSTAT"],
    location="midatl",
    epiweeks=EpiRange(202001, 202010),
).df()

# NoroSTAT metadata
epidata.pvt_meta_norostat(auth=os.environ["SECRET_API_AUTH_NOROSTAT"]).df()

# PAHO Dengue Sensors: Digital dengue surveillance
epidata.pvt_dengue_sensors(
    auth=os.environ["SECRET_API_AUTH_DENGUE_SENSORS"],
    names="ght",
    locations="ca",
    epiweeks=EpiRange(202001, 202010),
).df()

# Quidel: Influenza testing
epidata.pvt_quidel(
    auth=os.environ["SECRET_API_AUTH_QUIDEL"],
    locations="hhs1",
    epiweeks=EpiRange(202001, 202010),
).df()
```